In [ ]:
import numpy as np
import potcorr
import utli
import lab

import matplotlib.pyplot as plt
import const

In [ ]:
cell=lab.mos2_6x6_z24
ttkw = potcorr.PotCorr(cell)
ttkw.fft_init()
ttkw.get_vcoul()

In [ ]:
rho_tot.shape

In [ ]:
rho_dn = utli.read_dat("/anvil/scratch/x-rg47749/mos2/z/24/Rho_dn.dat")
rho_dc = utli.read_dat("/anvil/scratch/x-rg47749/mos2/z/24-spin/Rho_d.dat")

In [ ]:
rho_tot = rho_dc-rho_dn

In [ ]:
eps1 = '/anvil/scratch/x-rg47749/mos2/bgw_results/6x6/20/chimat.h5'
eps0 = '/anvil/scratch/x-rg47749/mos2/bgw_results/6x6/20/chi0mat.h5'
ttkw.read_epsinv(eps1=eps1, eps0=eps0)

In [ ]:
k_symmetry_map =  ttkw.get_k_symmetry_map()

In [ ]:
epsym_dict = ttkw.get_epsym_dict(k_symmetry_map,ecut=10)

In [ ]:
epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, epsmat_dict = ttkw.gen_mat_dict(k_symmetry_map, epsym_dict)

In [ ]:
ttkw.vcoul2d0modify()

In [ ]:
dpot_k = np.fft.fftn(rho_tot)*ttkw.v_coul2d

In [ ]:
rho_induced_G_dict = ttkw.gen_phi_G_dict(dpot_k, k_symmetry_map, epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, epsmat_dict)

In [ ]:
rho_induced_k = ttkw.map_phi_G(rho_induced_G_dict, epsmat_eps2rho_dict)

In [ ]:
rho_induced_r = np.fft.ifftn(rho_induced_k).real

In [ ]:
plt.plot(np.arange(225),np.sum(rho_tot,axis=(1,0)))

In [ ]:
np.sum(rho_tot)

In [ ]:
rho_ext = rho_tot-rho_induced_r

In [ ]:
rho_ext = np.load('/anvil/scratch/x-rg47749/mos2/z/22/rho_bare.npy')

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()*const.Bohr_R
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()*const.Bohr_R
fig, ax = plt.subplots(figsize=(2,2),dpi=600)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=np.sum(rho_tot[:,:,ttkw.fft_nz//2-5:ttkw.fft_nz//2+5], axis=(2)).real, marker='h',s=1,
                #gridsize=100,
                vmax=0.3,
                vmin=-0.3,
                    cmap='bwr')
#plt.axhline(y=4.709*np.sqrt(3), color='r', linestyle='--', linewidth=0.7)
#plt.axvline(x=4.709, color='r', linestyle='--', linewidth=0.7)
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(4.709-8.16, 4.709+8.16)
#ax.set_ylim(4.709*np.sqrt(3)-8.16, 4.709*np.sqrt(3)+8.16)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.025)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
rho_bare_z20 = np.load('/anvil/scratch/x-rg47749/mos2/z/20/rho_bare.npy')
rho_bare_z22 = np.load('/anvil/scratch/x-rg47749/mos2/z/22/rho_bare.npy')
rho_bare_z24 = np.load('/anvil/scratch/x-rg47749/mos2/z/24/rho_bare.npy')
rho_bare_z26 = np.load('/anvil/scratch/x-rg47749/mos2/z/26/rho_bare.npy')
rho_bare_z28 = np.load('/anvil/scratch/x-rg47749/mos2/z/28/rho_bare.npy')
rho_bare_z36 = np.load('/anvil/scratch/x-rg47749/mos2/z/36/rho_bare.npy')

In [ ]:
rho_avg = 1/4*(rho_bare_z20[:,:,192//2]+rho_bare_z22[:,:,216//2]+rho_bare_z28[:,:,270//2]+rho_bare_z36[:,:,360//2])

In [ ]:
fig, ax = plt.subplots(figsize=(3,2),dpi=600)
plt.plot(np.arange(180)*3.186*6/180,1/(const.Bohr_R**3)*rho_bare_z20[:,90,192//2],ls='-',label='z=20 $\\mathrm{\\AA}$')
plt.plot(np.arange(180)*3.186*6/180,1/(const.Bohr_R**3)*rho_bare_z22[:,90,216//2],ls='-',label='z=24 $\\mathrm{\\AA}$')
#plt.plot(np.arange(180)*3.186*6/180,rho_bare_z24[:,98,225//2+5],'-',label='z 24')
plt.plot(np.arange(180)*3.186*6/180,1/(const.Bohr_R**3)*rho_bare_z28[:,90,270//2],ls='-',lw=1.5,label='z=28 $\\mathrm{\\AA}$')
#plt.plot(np.arange(180)*3.186*6/180,rho_bare_z26[:,98,243//2+5],'-',label='z 32')
plt.plot(np.arange(180)*3.186*6/180,1/(const.Bohr_R**3)*rho_avg[:,90],ls='-',lw=1.5,label='z=32 $\\mathrm{\\AA}$')
plt.plot(np.arange(180)*3.186*6/180,1/(const.Bohr_R**3)*rho_bare_z36[:,90,360//2],ls= '-',label='z=36 $\\mathrm{\\AA}$')
plt.xlabel('x ($\\mathrm{\\AA}$)')
plt.ylabel(r'Charge Density ($\mathrm{\AA}^{-3}$)')

plt.legend(fontsize=5)

In [ ]:
np.sum(rho_bare_z20)*ttkw.omega/192/180/180

In [ ]:
3.186*3.186*6*6*20*np.sqrt(3)/2/(const.Bohr_R)**3

In [ ]:
ttkw.omega

In [ ]:
plt.plot(np.arange(180),(rho_bare_z20[:,98,192//2+5]-rho_bare_z20[:,98,192//2+5]),'-',label='z 20')
#plt.plot(np.arange(180),(rho_bare_z22[:,98,216//2+5]-rho_avg),'-',label='z 22')
plt.plot(np.arange(180),(rho_bare_z24[:,98,225//2+5]-rho_bare_z20[:,98,192//2+5]),'-',label='z 24')
plt.plot(np.arange(180),(rho_bare_z28[:,98,270//2+5]-rho_bare_z20[:,98,192//2+5]),'-',label='z 28')
plt.plot(np.arange(180),(rho_bare_z26[:,98,243//2+5]-rho_bare_z20[:,98,192//2+5]),'-',label='z 32')
plt.plot(np.arange(180),(rho_bare_z36[:,98,360//2+5]-rho_bare_z20[:,98,192//2+5]),'-',label='z 36')
#plt.plot(np.arange(180),rho_avg,'-',label='avg')
plt.legend(fontsize=5)

In [ ]:
dist_arr_tot = utli.distance_array_2d(180, 180,a = 3.186*6,b=3.186*6, defect_loc='center')

fig, ax = plt.subplots(figsize=(6,3),dpi=300)


ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z20[:,:,192//2].flatten().real,alpha=1, label = 'z=20 $\\mathrm{\\AA}$' ,s=20,marker='o')
ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z22[:,:,216//2].flatten().real,alpha=1, label = 'z=24 $\\mathrm{\\AA}$' ,s=20,marker='o')
#ax.scatter(dist_arr_tot.flatten(), rho_bare_z24[:,:,225//2].flatten().real,alpha=1, label = '24' ,s=80)
ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z28[:,:,270//2].flatten().real,alpha=1 , label = 'z=28 $\\mathrm{\\AA}$',s=20,marker='o')
#ax.scatter(dist_arr_tot.flatten(), rho_bare_z26[:,:,243//2].flatten().real,alpha=1 , label = '32',s=80)\
ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_avg[:,:].flatten().real,alpha=1 , label = 'z=32 $\\mathrm{\\AA}$',s=20,marker='o')
ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z36[:,:,360//2].flatten().real,alpha=1 , label = 'z=36 $\\mathrm{\\AA}$',s=20,marker='o')



#ax.scatter(dist_arr_tot.flatten(), rho_ext_12_m[:,:,ttkw.fft_nz//2].flatten().real,alpha=0.4)

plt.xlabel('Distance from Defect Center ($\\mathrm{\\AA}$)')
plt.ylabel(r'Charge Density ($\mathrm{\AA}^{-3}$)')

#plt.ylim(-0.0025,0.0025)
plt.legend()
plt.show()

In [ ]:
dist_arr_tot = utli.distance_array_2d(180, 180,a = 3.186*6,b=3.186*6, defect_loc='center')

fig, ax = plt.subplots(figsize=(4,3.5),dpi=300)


ax.scatter(1/(const.Bohr_R**3)*rho_avg[:,:].flatten().real, 1/(const.Bohr_R**3)*rho_bare_z20[:,:,192//2].flatten().real,alpha=1, label = 'z=20 $\\mathrm{\\AA}$' ,s=50,marker='o')
ax.scatter(1/(const.Bohr_R**3)*rho_avg[:,:].flatten().real, 1/(const.Bohr_R**3)*rho_bare_z22[:,:,216//2].flatten().real,alpha=1, label = 'z=24 $\\mathrm{\\AA}$' ,s=40,marker='o')
ax.scatter(1/(const.Bohr_R**3)*rho_avg[:,:].flatten().real, 1/(const.Bohr_R**3)*rho_bare_z28[:,:,270//2].flatten().real,alpha=1, label = 'z=28 $\\mathrm{\\AA}$' ,s=30,marker='o')
ax.scatter(1/(const.Bohr_R**3)*rho_avg[:,:].flatten().real, 1/(const.Bohr_R**3)*rho_avg[:,:].flatten().real,alpha=1, label = 'z=32 $\\mathrm{\\AA}$' ,s=20,marker='o')
ax.scatter(1/(const.Bohr_R**3)*rho_avg[:,:].flatten().real, 1/(const.Bohr_R**3)*rho_bare_z36[:,:,360//2].flatten().real,alpha=1, label = 'z=36 $\\mathrm{\\AA}$' ,s=10,marker='o')
#ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z22[:,:,216//2].flatten().real,alpha=1, label = 'z=24 $\\mathrm{\\AA}$' ,s=20,marker='o')
#ax.scatter(dist_arr_tot.flatten(), rho_bare_z24[:,:,225//2].flatten().real,alpha=1, label = '24' ,s=80)
#ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z28[:,:,270//2].flatten().real,alpha=1 , label = 'z=28 $\\mathrm{\\AA}$',s=20,marker='o')
#ax.scatter(dist_arr_tot.flatten(), rho_bare_z26[:,:,243//2].flatten().real,alpha=1 , label = '32',s=80)\
#ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_avg[:,:].flatten().real,alpha=1 , label = 'z=32 $\\mathrm{\\AA}$',s=20,marker='o')
#ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z36[:,:,360//2].flatten().real,alpha=1 , label = 'z=36 $\\mathrm{\\AA}$',s=20,marker='o')



#ax.scatter(dist_arr_tot.flatten(), rho_ext_12_m[:,:,ttkw.fft_nz//2].flatten().real,alpha=0.4)
plt.plot(np.linspace(-0.1,0.2,100),np.linspace(-0.1,0.2,100),c='black',ls='--',zorder = 0)

plt.xlabel(r'Averaged Charge Density ($\mathrm{\AA}^{-3}$)')
plt.ylabel(r'Charge Density with different z-size ($\mathrm{\AA}^{-3}$)')

plt.xlim(-0.04,0.2)
plt.ylim(-0.04,0.2)
#plt.axis('equal')
common_min = -0.04
common_max = 0.16
common_ticks = np.arange(common_min, common_max+0.04 , 0.04)
plt.xticks(common_ticks)
plt.yticks(common_ticks) # 设置相同的Y轴刻度
plt.legend(fontsize=7)
plt.show()

In [ ]:
dist_arr_tot = utli.distance_array_2d(180, 180,a = 3.186*6,b=3.186*6, defect_loc='center')

In [ ]:
dist_arr_tot.shape

In [ ]:
def coarse_grain_2d_numpy(array, block_shape, aggregation_func=np.mean, crop=True):
    """
    对2D NumPy数组进行粗粒化。

    参数:
    ----------
    array : numpy.ndarray
        输入的2D数组。
    block_shape : tuple of int
        一个包含两个整数的元组 (block_rows, block_cols)，
        定义了用于聚合的块的大小。
    aggregation_func : function, optional
        用于聚合每个块内元素的函数 (默认为 numpy.mean)。
        其他常用函数包括 numpy.sum, numpy.max, numpy.min, numpy.median。
    crop : bool, optional
        如果为 True (默认)，并且数组维度不能被 block_shape 整除，
        则会裁剪数组的边缘以使其能够被整除。
        如果为 False，且维度不能整除，则会引发 ValueError。

    返回:
    -------
    numpy.ndarray
        粗粒化后的新数组。

    异常:
    ------
    ValueError
        如果 crop=False 且数组维度不能被 block_shape 整除。
        如果 crop=True 但裁剪后的维度为0。
    """
    array = np.asarray(array)
    if array.ndim != 2:
        raise ValueError("输入数组必须是2维的。")

    orig_rows, orig_cols = array.shape
    block_rows, block_cols = block_shape

    if block_rows <= 0 or block_cols <= 0:
        raise ValueError("block_shape 的维度必须是正整数。")

    if crop:
        # 裁剪数组以适应块大小
        target_rows = (orig_rows // block_rows) * block_rows
        target_cols = (orig_cols // block_cols) * block_cols
        if target_rows == 0 or target_cols == 0:
            raise ValueError(
                f"数组维度 ({orig_rows}, {orig_cols}) 对于块大小 ({block_rows}, {block_cols}) "
                f"来说太小，无法进行裁剪和粗粒化。"
            )
        array_to_process = array[:target_rows, :target_cols]
    else:
        if orig_rows % block_rows != 0 or orig_cols % block_cols != 0:
            raise ValueError(
                "当 crop=False 时，数组维度必须能被 block_shape 整除。"
                "考虑使用 crop=True 或预先填充数组。"
            )
        array_to_process = array

    # 计算新数组的维度
    new_num_rows = array_to_process.shape[0] // block_rows
    new_num_cols = array_to_process.shape[1] // block_cols

    # 关键步骤：
    # 1. 将数组重塑为 (new_num_rows, block_rows, new_num_cols, block_cols)
    #    这使得每个 (block_rows, block_cols) 子块在逻辑上是连续的。
    reshaped = array_to_process.reshape(new_num_rows, block_rows, new_num_cols, block_cols)

    # 2. 交换轴，使得属于同一个输出单元格的块元素在最后两个维度
    #    形状变为 (new_num_rows, new_num_cols, block_rows, block_cols)
    transposed = reshaped.transpose(0, 2, 1, 3)

    # 3. 再次重塑，将每个块展平为一个向量
    #    形状变为 (new_num_rows, new_num_cols, block_rows * block_cols)
    #    现在，每个粗粒化单元的所有原始值都在最后一个维度中。
    blocks_as_vectors = transposed.reshape(new_num_rows, new_num_cols, block_rows * block_cols)

    # 4. 沿着最后一个轴应用聚合函数
    coarse_grained_array = aggregation_func(blocks_as_vectors, axis=2)

    return coarse_grained_array

In [ ]:
rho_20_co = coarse_grain_2d_numpy(rho_bare_z20[:,:,192//2],(6,6), aggregation_func=np.mean, crop=True)


In [ ]:
dist_arr_tot = utli.distance_array_2d(30, 30,a = 3.186*6,b=3.186*6, defect_loc='center')

fig, ax = plt.subplots(figsize=(6,3),dpi=300)


ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_20_co.flatten().real,alpha=1, label = 'z=20 $\\mathrm{\\AA}$' ,s=20,marker='o')
#ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z22[:,:,216//2].flatten().real,alpha=1, label = 'z=24 $\\mathrm{\\AA}$' ,s=20,marker='o')
#ax.scatter(dist_arr_tot.flatten(), rho_bare_z24[:,:,225//2].flatten().real,alpha=1, label = '24' ,s=80)
#ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z28[:,:,270//2].flatten().real,alpha=1 , label = 'z=28 $\\mathrm{\\AA}$',s=20,marker='o')
#ax.scatter(dist_arr_tot.flatten(), rho_bare_z26[:,:,243//2].flatten().real,alpha=1 , label = '32',s=80)\
#ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_avg[:,:].flatten().real,alpha=1 , label = 'z=32 $\\mathrm{\\AA}$',s=20,marker='o')
#ax.scatter(dist_arr_tot.flatten(), 1/(const.Bohr_R**3)*rho_bare_z36[:,:,360//2].flatten().real,alpha=1 , label = 'z=36 $\\mathrm{\\AA}$',s=20,marker='o')



#ax.scatter(dist_arr_tot.flatten(), rho_ext_12_m[:,:,ttkw.fft_nz//2].flatten().real,alpha=0.4)

plt.xlabel('Distance from Defect Center ($\\mathrm{\\AA}$)')
plt.ylabel(r'Charge Density ($\mathrm{\AA}^{-3}$)')

#plt.ylim(-0.0025,0.0025)
plt.legend()
plt.show()